# Week 11 — Wednesday: Ensembles

**DATA 202 · Calvin University**

<img src="https://cs.calvin.edu/courses/data/202/26fa/weeks/11/images/pumpkins.jpg" width="420" alt="A pile of small orange pumpkins on a bale of straw.">

In Turkey, roasted pumpkin seeds are a favorite snack. Two varieties: **Çerçevelik** and **Ürgüp Sivrisi** ("the pointy one from Ürgüp"). Researchers photographed 2,500 seeds and measured each seed's shape: 12 columns, such as length, width, area and aspect ratio (length ÷ width).

Monday's model, logistic regression, draws a straight boundary between the two varieties and gets 88.9% in cross-validation. Trees can draw bendy boundaries, but one deep tree overfits (Week 7).

> 🎯 **How can many models, combined, beat one?**

**Today's plan (~40 min):**

| Time | Part |
|---|---|
| ~3 min | 1 · One tree, no limit |
| ~8 min | 2 · Bagging: random forests (SLO 11B) |
| ~5 min | 3 · Why bagging doesn't overfit (SLO 11B) |
| ~10 min | 4 · Boosting: AdaBoost (SLO 11B) |
| ~5 min | 5 · Stacking (SLO 11B) |
| ~4 min | 6 · The final comparison |
| ~5 min | 7 · The value of diversity |

**Cues:** 🎯 Predict First · 💬 Question · 🗣️ Pair Talk · 📌 One split

<sub>Data: Koklu, Sarigil & Ozbek (2021), *Genetic Resources and Crop Evolution* 68(7), 2713–2726. Photo: Unsplash</sub>

---
## 1 · One Tree, No Limit · ~3 min

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.model_selection import train_test_split, cross_val_score, cross_validate
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, AdaBoostClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

seeds = pd.read_csv("https://cs.calvin.edu/courses/data/202/26fa/datasets/pumpkin.csv")
X = seeds.drop(columns="variety")
y = seeds["variety"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1)
print(len(X_train), "training seeds |", len(X_test), "test seeds")

tree = DecisionTreeClassifier(random_state=1)
scores = cross_validate(tree, X_train, y_train, cv=5, return_train_score=True)
print(f"one tree, no depth limit | folds it learned from: {scores['train_score'].mean():.3f}"
      f" | held-out fold: {scores['test_score'].mean():.3f}")
tree.fit(X_train, y_train)
print(f"{tree.get_depth()} levels deep, {tree.get_n_leaves()} leaves")

> 💬 **Question:** What is this tree doing? What did we do about it in Week 7?

<details><summary>Answer</summary>

* 100% on the seeds it learned from, 83.0% on held-out seeds, with 195 leaves: it memorized. **Overfitting**
* Week 7's fix: limit the depth. The best depth here is 4, at 87.2%, still below logistic regression's 88.9%
* Today's fix: combine **many** models into one, an **ensemble**
</details>

Three recipes for an ensemble, one per part of today's class:

<img src="https://cs.calvin.edu/courses/data/202/26fa/weeks/11/images/ensembles.png" width="720" alt="Three diagrams. Bagging: different subsets of the training data each train a base model, and the most frequent or average prediction wins. Boosting: base models in a row, each trained on data modified by the previous model's predictions. Stacking: different models whose predictions feed a meta-learner.">

---
## 2 · Bagging: Random Forests (SLO 11B) · ~8 min

**Bagging** = **b**ootstrap **agg**regat**ing**:

1. Draw a **bootstrap sample**: as many seeds as the training set has, drawn *with replacement*
2. Grow one deep tree on each sample, so every tree is a little different
3. **Aggregate:** to predict, the trees **vote**, and the majority wins

In [ ]:
rng = np.random.default_rng(3)
print("ten seeds:           ", list(range(1, 11)))
print("a bootstrap sample:  ", sorted(rng.choice(range(1, 11), size=10, replace=True)))

* Some seeds come twice, some not at all. On average, a bootstrap sample leaves out about a third of the seeds

A **random forest** is bagging with one more twist: at each split, a tree may look at only a **random few** of the 12 columns. Its trees differ even more.

In [ ]:
bagging = BaggingClassifier(DecisionTreeClassifier(), n_estimators=100, random_state=1)   # plain bagging of trees
forest = RandomForestClassifier(n_estimators=100, random_state=1)                          # bagging + random columns
for name, model in [("bagging, 100 trees", bagging), ("random forest, 100 trees", forest)]:
    print(f"{name:24} | 5-fold CV accuracy: {cross_val_score(model, X_train, y_train, cv=5).mean():.3f}")

Bagging: 88.0%. Random forest: 88.3%. Both far above the single deep tree (83.0%). How does the vote do it? A small forest of 5 trees, on 6 test seeds:

In [ ]:
forest5 = RandomForestClassifier(n_estimators=5, random_state=1).fit(X_train, y_train)

six = X_test.loc[[2368, 1276, 297, 679, 1561, 655]]
votes = pd.DataFrame(index=["A", "B", "C", "D", "E", "F"])
for i, one_tree in enumerate(forest5.estimators_):
    # the trees inside a forest answer 0 or 1; forest5.classes_ turns that back into names
    votes[f"tree {i + 1}"] = forest5.classes_[one_tree.predict(six.to_numpy()).astype(int)]
votes

> 📄 **Handout 1 — Bagging: five trees vote.** Count the votes and fill in the forest's answer. Then check it against the actual variety. 🗣️ Which single tree would you trust alone?

In [ ]:
votes["forest"] = forest5.predict(six)
votes["actual"] = y_test[six.index].to_numpy()
votes

<details><summary>Answer</summary>

* The forest gets **5 of 6** right. Every single tree gets only 3 or 4: each one slips on a different seed, and the others outvote it
* No tree is reliable alone. The **vote** is
* **Seed F** fools all five trees. A vote can't fix a mistake every voter makes
</details>

---
## 3 · Why Bagging Doesn't Overfit (SLO 11B) · ~5 min

Same knob as Week 7, `max_depth`, for one tree and for a forest of 100. (About 30 seconds to run.)

In [ ]:
rows = []
for depth in [1, 2, 3, 4, 6, 8, 10, 15, None]:
    for name, model in [("one tree", DecisionTreeClassifier(max_depth=depth, random_state=1)),
                        ("forest of 100", RandomForestClassifier(max_depth=depth, random_state=1, n_jobs=-1))]:
        scores = cross_validate(model, X_train, y_train, cv=5, return_train_score=True)
        rows.append({"max_depth": str(depth), "model": name, "scored on": "folds it learned from",
                     "accuracy": scores["train_score"].mean()})
        rows.append({"max_depth": str(depth), "model": name, "scored on": "held-out fold",
                     "accuracy": scores["test_score"].mean()})
fig = px.line(pd.DataFrame(rows), x="max_depth", y="accuracy", color="model", line_dash="scored on", markers=True,
              title="One Tree vs. a Forest, as the Trees Grow Deeper")
fig.show()

> 📄 **Handout 2 — Tree vs. forest.** Label the tree's overfitting side. 🗣️ Both reach 100% on the seeds they learned from. Why doesn't the forest's held-out accuracy fall?

<details><summary>Answer</summary>

* **One tree:** best at depth 4 (87.2%), then down to **83.0%** with no limit: overfitting
* **Forest:** about **88%** from depth 4 on, and still **88.3%** with no limit
* Each deep tree memorizes its own bootstrap sample, with its own quirks. The quirks disagree and get outvoted; the real pattern (long seeds are Ürgüp Sivrisi) is shared, and wins the vote
* Many overfit trees, voting, overfit much less. So a forest rarely needs a depth limit, and more trees never hurt
</details>

---
## 4 · Boosting: AdaBoost (SLO 11B) · ~10 min

Bagging grows its trees side by side, each on its own. **Boosting** builds models **in a row**: each new one focuses on what the ones before it got wrong. **AdaBoost** (*adaptive boosting*, by Yoav Freund and Robert Schapire, 1990s):

1. Every training seed starts with the same **weight**
2. Fit a **stump**: a tree with a single question
3. The stump's **error** = the total weight of the seeds it gets wrong. Its **say** = $\frac{1}{2}\ln\frac{1 - \text{error}}{\text{error}}$: the more accurate, the bigger its say
4. **Reweight:** the seeds it got wrong get heavier, the rest lighter
5. Back to step 2 with the new weights. To predict, the stumps vote, each with its say

By hand first, on 20 training seeds and two columns: shape (aspect ratio) and size (area).

In [ ]:
few = X_train.sample(20, random_state=73)[["aspect_ratio", "area"]]       # 20 seeds, two columns
few_variety = y_train[few.index]
weight = np.full(20, 1 / 20)                                               # step 1: every seed weighs the same

stump1 = DecisionTreeClassifier(max_depth=1).fit(few, few_variety, sample_weight=weight)   # step 2
wrong1 = (stump1.predict(few) != few_variety).to_numpy()
error1 = weight[wrong1].sum()                                              # step 3
say1 = 0.5 * np.log((1 - error1) / error1)
print(export_text(stump1, feature_names=["aspect_ratio", "area"]))
print(f"stump 1 | wrong: {wrong1.sum()} seeds | error {error1:.2f} | say {say1:.2f}")

fig = px.scatter(few, x="aspect_ratio", y="area", color=few_variety, symbol=few_variety,
                 labels={"aspect_ratio": "aspect ratio (length ÷ width)", "area": "area (pixels)", "color": "", "symbol": ""},
                 title="Stump 1 on 20 Seeds")
fig.update_traces(marker_size=11)
fig.add_vline(x=stump1.tree_.threshold[0], line_dash="dash")              # the stump's one cut
fig.show()

> 📄 **Handout 3 — AdaBoost by hand, question 1.** Circle the seeds stump 1 gets wrong.

Step 4, reweight:

In [ ]:
weight = weight * np.where(wrong1, np.exp(say1), np.exp(-say1))    # wrong seeds heavier, right seeds lighter
weight = weight / weight.sum()                                     # the weights add up to 1 again
print(f"each seed stump 1 got wrong now weighs {weight[wrong1][0]:.3f}; each one it got right, {weight[~wrong1][0]:.3f}")

fig = px.scatter(few, x="aspect_ratio", y="area", color=few_variety, symbol=few_variety, size=weight,
                 labels={"aspect_ratio": "aspect ratio (length ÷ width)", "area": "area (pixels)", "color": "", "symbol": ""},
                 title="Round 2: Each Seed Drawn by Its New Weight")
fig.show()

> 📄 **Handout 3, question 2.** 🎯 Draw the single cut (one vertical or one horizontal line) that stump 2 should make.

In [ ]:
stump2 = DecisionTreeClassifier(max_depth=1).fit(few, few_variety, sample_weight=weight)
wrong2 = (stump2.predict(few) != few_variety).to_numpy()
error2 = weight[wrong2].sum()
say2 = 0.5 * np.log((1 - error2) / error2)
print(export_text(stump2, feature_names=["aspect_ratio", "area"]))
print(f"stump 2 | wrong: {wrong2.sum()} seeds | error {error2:.2f} | say {say2:.2f}")

fig.add_hline(y=stump2.tree_.threshold[0], line_dash="dash")
fig.show()

One more round, then step 5: the stumps vote, each with its say.

In [ ]:
weight = weight * np.where(wrong2, np.exp(say2), np.exp(-say2))    # step 4 again
weight = weight / weight.sum()
stump3 = DecisionTreeClassifier(max_depth=1).fit(few, few_variety, sample_weight=weight)
wrong3 = (stump3.predict(few) != few_variety).to_numpy()
error3 = weight[wrong3].sum()
say3 = 0.5 * np.log((1 - error3) / error3)
print(export_text(stump3, feature_names=["aspect_ratio", "area"]))
print(f"stump 3 | wrong: {wrong3.sum()} seeds | error {error3:.2f} | say {say3:.2f}")

# step 5: each stump votes +1 (Ürgüp Sivrisi) or -1 (Çerçevelik), times its say
total = np.zeros(20)
for stump, say in [(stump1, say1), (stump2, say2), (stump3, say3)]:
    total = total + say * np.where(stump.predict(few) == "Ürgüp Sivrisi", 1, -1)
vote = np.where(total > 0, "Ürgüp Sivrisi", "Çerçevelik")
print(f"\nthe weighted vote of the 3 stumps: {(vote == few_variety).sum()} of 20 right")

missed = wrong1.astype(int) + wrong2.astype(int) + wrong3.astype(int)   # how many stumps miss each seed
print("seeds missed by 0, 1, 2, 3 stumps:", np.bincount(missed, minlength=4))

> 📄 **Handout 3, question 3.** 🗣️ Each stump alone misses several seeds, yet their weighted vote gets all 20 right. How?

<details><summary>Answer</summary>

* **Stump 1** asks "aspect ratio ≤ 2.12?" and misses **3** seeds: Ürgüp Sivrisi that are unusually round, and big. Error 3 × 0.05 = 0.15, say **0.87**
* **Reweighting:** each of the 3 now weighs **0.167**, each of the other 17 only **0.029**. Together the 3 hold half the weight
* **Stump 2** asks a different question, "area ≤ 93,591?", and gets all 3 heavy seeds right. It misses 4 small, light ones instead: weighted error 0.12, say **1.01**
* **Stump 3** goes back to aspect ratio (≤ 1.85, say **1.10**) and misses 6
* **No seed is missed by two stumps.** Each stump was trained to get right what the ones before it missed, so their mistakes don't line up, and the other two always outvote the one that slips
</details>

`AdaBoostClassifier` does exactly this, round after round, on all 12 columns:

In [ ]:
for n in [1, 10, 30, 100, 300]:
    acc = cross_val_score(AdaBoostClassifier(n_estimators=n, random_state=1), X_train, y_train, cv=5).mean()
    print(f"AdaBoost, {n:3} stumps | 5-fold CV accuracy: {acc:.3f}")

<details><summary>Answer</summary>

* 1 stump: **86.7%**. A single question about the aspect ratio already gets most seeds right
* Each new stump adds a little: **87.8%** with 300 stumps
* Bagging's trees are **independent** voters, each deep. Boosting's stumps are **weak** alone and build on each other
* Boosting's knob is `n_estimators`. On these seeds, more stumps barely overfit: with 1,000, still 88.1% held out (90.6% on the folds it learned from). Boosting can overfit when some labels are wrong, because a mislabeled seed gets heavier every round. Choose `n_estimators` with cross-validation
* The other famous family, **gradient boosting** (XGBoost, LightGBM), also builds trees in a row, each fit to what the trees before it still get wrong
</details>

---
## 5 · Stacking (SLO 11B) · ~5 min

Bagging and boosting combine many copies of **one kind** of model. **Stacking** combines **different kinds**:

* Each **base model** (here: logistic regression, kNN, a random forest) predicts a probability for every seed
* A **meta-learner**, here a logistic regression, takes those probabilities as its columns and learns **how much to trust each model**
* It learns from **cross-validated** predictions, so a base model that just memorized its seeds can't fool it

(About 10 seconds to run.)

In [ ]:
stack = StackingClassifier(
    estimators=[("logistic", make_pipeline(StandardScaler(), LogisticRegression(C=100, max_iter=10_000))),
                ("knn", make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15))),
                ("forest", RandomForestClassifier(random_state=1))],
    final_estimator=LogisticRegression(), cv=5)
print(f"stacking | 5-fold CV accuracy: {cross_val_score(stack, X_train, y_train, cv=5).mean():.3f}")

stack.fit(X_train, y_train)
pd.Series(stack.final_estimator_.coef_[0], index=["logistic", "knn", "forest"], name="meta-learner's slope").round(2)

> 💬 **Question:** Which base model does the meta-learner trust most? Why might it give kNN the smallest slope?

<details><summary>Answer</summary>

* **88.8%** in cross-validation: no better than logistic regression alone (88.9%)
* The meta-learner's slopes: logistic regression **3.08**, forest **2.48**, kNN **0.87**. A bigger slope means that model's probability moves the final answer more
* kNN (88.1% alone) is about as accurate as the others, but its mistakes overlap the most with the logistic regression's, so it adds little new
* Why no gain? The three mostly miss the **same** seeds: in cross-validation, **159** training seeds fool all three. A meta-learner can't fix a mistake every member makes (seed F again)
</details>

---
## 6 · The Final Comparison · ~4 min

Each model with its knob chosen by cross-validation (for logistic regression, a grid over C like Monday's picks C = 100). Then, once, the locked test set. (About 30 seconds to run.)

In [ ]:
finalists = {
    "logistic regression, C = 100": make_pipeline(StandardScaler(), LogisticRegression(C=100, max_iter=10_000)),
    "one tree, depth 4": DecisionTreeClassifier(max_depth=4, random_state=1),
    "bagging: random forest": RandomForestClassifier(random_state=1),
    "boosting: AdaBoost, 300 stumps": AdaBoostClassifier(n_estimators=300, random_state=1),
    "stacking": stack,
}
rows = []
for name, model in finalists.items():
    cv = cross_val_score(model, X_train, y_train, cv=5).mean()
    test = model.fit(X_train, y_train).score(X_test, y_test)
    rows.append({"model": name, "5-fold CV": round(cv, 3), "test": round(test, 3)})
pd.DataFrame(rows)

> 💬 **Question:** Which model would you ship?

<details><summary>Answer</summary>

* CV ranks logistic regression first (88.9%) and stacking second (88.8%); the test set ranks the forest first (89.1%). All the ensembles land within about a point of each other
* 📌 **One split:** on 625 test seeds, one point is about 6 seeds. The order flips between CV and test, so the differences are mostly luck
* With a tie, prefer the simpler model: the logistic regression, whose slopes we can read
* The ensembles' real win is over the **single** tree: from 83% up to about 88%
</details>

---
## 7 · The Value of Diversity · ~5 min

Is it the *differences* between the members that make an ensemble work? Part 2's bagging again, with one change, `bootstrap=False`: all 100 trees learn from the same 1,875 seeds.

> 🎯 **Predict First:** what will these 100 trees score?

In [ ]:
clones = BaggingClassifier(DecisionTreeClassifier(), n_estimators=100, bootstrap=False, random_state=1)   # no bootstrap samples
print(f"bagging, 100 trees, all on the same seeds | 5-fold CV accuracy: {cross_val_score(clones, X_train, y_train, cv=5).mean():.3f}")

<details><summary>Answer</summary>

* **83.1%**: the same as one tree (83.0%), far below real bagging (88.0%). 100 copies of one tree vote exactly like that tree
* An ensemble gains only from members that are **different**, so each recipe builds the differences in:

| recipe | the members differ in |
|---|---|
| bagging, random forest | the seeds each tree learns from (and the columns it may use) |
| boosting | what each stump focuses on: the seeds the ones before it missed |
| stacking | the kind of model |
</details>

And the mistakes that every member shares?

In [ ]:
wrong = pd.DataFrame({name: model.predict(X_test) != y_test for name, model in finalists.items()})
print(f"test seeds that all five finalists get wrong: {wrong.all(axis=1).sum()} of {len(y_test)}")

<details><summary>Answer</summary>

* **47** seeds (8%) fool every model: they look like the other variety. That is the ceiling of what shape alone can tell
</details>

> I looked, and there before me was a great multitude that no one could count, from every nation, tribe, people and language, standing before the throne.
> — Revelation 7:9

> 💬 **Question:** 100 identical trees did no better than one. Seed F fooled all five trees, and 47 test seeds fool every model. Where have you seen a group whose members all shared the same blind spot? What does a different perspective add that one more of the same cannot?

<details><summary>Answer</summary>

* Agreement adds nothing: a copy only repeats what the group already says. The gain comes from members who see what the others miss, whose errors don't line up
* The flip side: if every member learned from the same skewed data, they share the same blind spot, and the vote keeps it (Week 14, fairness)
* Scripture pictures God's people as one body with many members (1 Cor 12; Rev 5:9; Rev 7:9): the differences are part of the design, not a defect to average away
</details>

---
### Wrap-up

1. One deep tree **overfits**. An **ensemble** combines many models into one
2. **Bagging** (a **random forest**): deep trees on bootstrap samples, plus random columns, then a majority vote. Their quirks cancel, so it overfits far less; more trees never hurt
3. **Boosting** (**AdaBoost**): stumps in a row. Each round makes the missed seeds heavier, and the stumps vote with their say. Choose `n_estimators` with CV
4. **Stacking**: different kinds of models, and a meta-learner that learns whom to trust
5. An ensemble gains only from **different** members: 100 identical trees score like one, and mistakes they all share survive the vote
6. On these seeds every good model lands near 88% 📌. The ensembles beat the single tree, not the simple line